# IR Assignment 2 — Notebook 04: Hybrid Fusion (R4 & R5)

Combines BM25F (R2) and Dense Retrieval (R3) scores using two fusion strategies:
- **R4** — Weighted sum: `score = α × norm_bm25f + (1−α) × norm_dense`
- **R5** — Reciprocal Rank Fusion (RRF): `score = 1/(k + rank_bm25f) + 1/(k + rank_dense)`

**Prerequisites:** Run `parser.py` to generate `parsed_patents.jsonl`, then run Notebook 03 to build the FAISS index.


In [ ]:
from pathlib import Path
import re
import json
import pickle

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from rank_bm25 import BM25Okapi


## 1 — Paths & Settings


In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
INTERIM_DIR  = PROJECT_ROOT / "data" / "interim"
RESULTS_DIR  = PROJECT_ROOT / "results" / "fusion"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PARSED_PATH = INTERIM_DIR / "parsed_patents.jsonl"
FAISS_PATH  = INTERIM_DIR / "patent_dense_mvp.index"
META_PATH   = INTERIM_DIR / "chunk_metadata_mvp.pkl"

# BM25F field weights (must match Notebook 02)
FIELD_WEIGHTS = {"title": 3.0, "abstract": 2.0, "claims": 1.5, "description": 1.0}
FIELDS        = list(FIELD_WEIGHTS.keys())
TOP_K         = 10
POOL_SIZE     = 100   # candidates per system before fusion

print("PARSED_PATH exists:", PARSED_PATH.exists())
print("FAISS_PATH  exists:", FAISS_PATH.exists())
print("META_PATH   exists:", META_PATH.exists())


## 2 — Load Data


In [ ]:
docs_df = pd.read_json(PARSED_PATH, lines=True)
docs_df["ucid"] = docs_df["ucid"].astype(str)
for field in FIELDS:
    docs_df[field] = docs_df[field].fillna("")

TOKEN_RE = re.compile(r"[A-Za-z0-9]+")
def tokenize(text: str):
    return TOKEN_RE.findall(text.lower())

print(f"Loaded {len(docs_df):,} patents")
docs_df[["ucid","title"]].head(3)


## 3 — Build BM25F Index (R2)


In [ ]:
for field in FIELDS:
    docs_df[f"tokens_{field}"] = docs_df[field].map(tokenize)

bm25_indexes = {}
for field in FIELDS:
    bm25_indexes[field] = BM25Okapi(docs_df[f"tokens_{field}"].tolist())
    print(f"BM25 index built for '{field}'")


In [ ]:
def search_bm25f(query_text: str, top_k: int = POOL_SIZE, exclude_ucid: str = None):
    """Returns a DataFrame with columns: ucid, bm25f_score, rank (top_k results)."""
    tokens = tokenize(query_text)
    if not tokens:
        raise ValueError("Empty query after tokenisation.")

    scores = np.zeros(len(docs_df))
    for field, weight in FIELD_WEIGHTS.items():
        scores += weight * bm25_indexes[field].get_scores(tokens)

    result_df = docs_df[["ucid", "title"]].copy()
    result_df["bm25f_score"] = scores

    if exclude_ucid:
        result_df = result_df[result_df["ucid"] != exclude_ucid]

    result_df = result_df.sort_values("bm25f_score", ascending=False).head(top_k).reset_index(drop=True)
    result_df.insert(0, "rank", range(1, len(result_df) + 1))
    return result_df


print("search_bm25f defined")


## 4 — Load Dense Index (R3)


In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

if not FAISS_PATH.exists() or not META_PATH.exists():
    raise FileNotFoundError(
        "FAISS index not found. Run Notebook 03 first to build it."
    )

faiss_index = faiss.read_index(str(FAISS_PATH))
with open(META_PATH, "rb") as f:
    chunk_metadata = pickle.load(f)

model = SentenceTransformer("anferico/bert-for-patents")
print(f"FAISS index loaded: {faiss_index.ntotal:,} vectors")
print(f"Chunk metadata: {len(chunk_metadata):,} chunks")


In [ ]:
def search_dense(query_text: str, top_n: int = POOL_SIZE, exclude_ucid: str = None):
    """Returns a dict {ucid: score} with MaxP aggregation."""
    query_emb = model.encode([query_text], normalize_embeddings=True)
    distances, indices = faiss_index.search(
        np.array(query_emb, dtype=np.float32), k=top_n * 5
    )

    patent_scores = {}
    for score, idx in zip(distances[0], indices[0]):
        ucid = chunk_metadata[idx]["ucid"]
        if exclude_ucid and ucid == exclude_ucid:
            continue
        if ucid not in patent_scores or score > patent_scores[ucid]:
            patent_scores[ucid] = float(score)

    ranked = sorted(patent_scores.items(), key=lambda x: x[1], reverse=True)
    return dict(ranked[:top_n])


print("search_dense defined")


## 5 — Fusion Functions


In [ ]:
def fuse_r4(query_text: str, alpha: float = 0.5, top_k: int = TOP_K,
            exclude_ucid: str = None):
    """
    R4 — Weighted sum fusion.
    score = alpha * norm(bm25f) + (1 - alpha) * norm(dense)
    Both score lists are min-max normalised to [0, 1] before combining.
    """
    bm25f_df = search_bm25f(query_text, top_k=POOL_SIZE, exclude_ucid=exclude_ucid)
    dense_d  = search_dense(query_text,  top_n=POOL_SIZE, exclude_ucid=exclude_ucid)

    bm25f_scores = dict(zip(bm25f_df["ucid"], bm25f_df["bm25f_score"]))
    dense_scores = dense_d

    # min-max normalise each system's scores
    def minmax(d):
        vals = list(d.values())
        lo, hi = min(vals), max(vals)
        rng = hi - lo if hi > lo else 1.0
        return {k: (v - lo) / rng for k, v in d.items()}

    bm25f_norm = minmax(bm25f_scores)
    dense_norm = minmax(dense_scores)

    all_ucids = set(bm25f_norm) | set(dense_norm)
    combined = {
        ucid: alpha * bm25f_norm.get(ucid, 0.0) + (1 - alpha) * dense_norm.get(ucid, 0.0)
        for ucid in all_ucids
    }

    ranked = sorted(combined.items(), key=lambda x: x[1], reverse=True)[:top_k]

    rows = []
    for rank, (ucid, score) in enumerate(ranked, 1):
        row = docs_df[docs_df["ucid"] == ucid]
        title = row.iloc[0]["title"] if not row.empty else ""
        rows.append({
            "rank":         rank,
            "ucid":         ucid,
            "r4_score":     round(score, 6),
            "bm25f_score":  round(bm25f_scores.get(ucid, 0.0), 4),
            "dense_score":  round(dense_scores.get(ucid, 0.0), 4),
            "title":        title,
        })
    return pd.DataFrame(rows)


print(f"fuse_r4 defined (default α=0.5)")


In [ ]:
def fuse_r5(query_text: str, k: int = 60, top_k: int = TOP_K,
            exclude_ucid: str = None):
    """
    R5 — Reciprocal Rank Fusion (RRF).
    score = 1/(k + rank_bm25f) + 1/(k + rank_dense)
    No score normalisation needed — only rank positions matter.
    """
    bm25f_df = search_bm25f(query_text, top_k=POOL_SIZE, exclude_ucid=exclude_ucid)
    dense_d  = search_dense(query_text,  top_n=POOL_SIZE, exclude_ucid=exclude_ucid)

    bm25f_ranks = {row["ucid"]: int(row["rank"]) for _, row in bm25f_df.iterrows()}
    dense_ranks = {ucid: i + 1 for i, ucid in enumerate(dense_d.keys())}

    all_ucids = set(bm25f_ranks) | set(dense_ranks)
    rrf_scores = {
        ucid: 1.0 / (k + bm25f_ranks.get(ucid, POOL_SIZE + 1))
            + 1.0 / (k + dense_ranks.get(ucid,  POOL_SIZE + 1))
        for ucid in all_ucids
    }

    ranked = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

    rows = []
    for rank, (ucid, score) in enumerate(ranked, 1):
        row = docs_df[docs_df["ucid"] == ucid]
        title = row.iloc[0]["title"] if not row.empty else ""
        rows.append({
            "rank":        rank,
            "ucid":        ucid,
            "rrf_score":   round(score, 6),
            "bm25f_rank":  bm25f_ranks.get(ucid, "-"),
            "dense_rank":  dense_ranks.get(ucid, "-"),
            "title":       title,
        })
    return pd.DataFrame(rows)


print(f"fuse_r5 defined (default k=60)")


## 6 — Demo: Free-Text Query (R2 vs R3 vs R4 vs R5)


In [ ]:
demo_query = "internal combustion engine fuel injection system"

print(f"Query: '{demo_query}'")
print("=" * 70)

r2 = search_bm25f(demo_query, top_k=TOP_K)
r3 = search_dense(demo_query,  top_n=TOP_K)
r4 = fuse_r4(demo_query, alpha=0.5, top_k=TOP_K)
r5 = fuse_r5(demo_query, k=60,      top_k=TOP_K)

print("\n--- R2: BM25F ---")
print(r2[["rank","ucid","bm25f_score","title"]].to_string(index=False))

print("\n--- R3: Dense ---")
dense_rows = [{"rank": i+1, "ucid": u, "dense_score": round(s,4)} for i,(u,s) in enumerate(r3.items())]
print(pd.DataFrame(dense_rows).to_string(index=False))

print("\n--- R4: Weighted Fusion (α=0.5) ---")
print(r4[["rank","ucid","r4_score","bm25f_score","dense_score","title"]].to_string(index=False))

print("\n--- R5: RRF (k=60) ---")
print(r5[["rank","ucid","rrf_score","bm25f_rank","dense_rank","title"]].to_string(index=False))


## 7 — R4: Tune α (Sparse vs Dense Weight)


In [ ]:
alphas = [0.2, 0.4, 0.5, 0.6, 0.8]

print(f"Query: '{demo_query}'\n")
print(f"{'alpha':>6}  {'Top-3 UCIDs'}")
print("-" * 70)
for alpha in alphas:
    res = fuse_r4(demo_query, alpha=alpha, top_k=3)
    top3 = " | ".join(res["ucid"].tolist())
    print(f"  {alpha:.1f}    {top3}")


## 8 — Save Results


In [ ]:
# Save R4 and R5 results for the demo query
r4_path = RESULTS_DIR / "r4_fusion_results.csv"
r5_path = RESULTS_DIR / "r5_rrf_results.csv"

r4.to_csv(r4_path, index=False)
r5.to_csv(r5_path, index=False)

# Save config
config = {
    "fusion_query":    demo_query,
    "field_weights":   FIELD_WEIGHTS,
    "r4_alpha":        0.5,
    "r5_k":            60,
    "pool_size":       POOL_SIZE,
    "top_k":           TOP_K,
    "dense_model":     "anferico/bert-for-patents",
}
(RESULTS_DIR / "fusion_config.json").write_text(json.dumps(config, indent=2))

print(f"R4 results → {r4_path}")
print(f"R5 results → {r5_path}")
print(f"Config     → {RESULTS_DIR / 'fusion_config.json'}")
